In [1]:
from gensim.models import Word2Vec

sentences = [
    ["the", "king", "is", "a", "man"],
    ["the", "queen", "is", "a", "woman"],
    ["the", "man", "is", "strong"],
    ["the", "woman", "is", "strong"],
    ["the", "king", "rules", "the", "kingdom"],
    ["the", "queen", "rules", "the", "kingdom"],
]

model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=1
)

king_vector = model.wv["king"]

print(king_vector)

[-0.00515491 -0.00666842 -0.00777572  0.00831149 -0.00198232 -0.00685586
 -0.00415473  0.00514378 -0.00286805 -0.00375057  0.00162284 -0.00277677
 -0.00158266  0.00107453 -0.00297686  0.0085203   0.00391191 -0.00995955
  0.00625866 -0.00675326  0.00076981  0.0044055  -0.00510475 -0.00211223
  0.00809603 -0.00424367 -0.00763639  0.00925766 -0.00215423 -0.00471984
  0.00856949  0.00428336  0.00432473  0.00928628 -0.00845427  0.00525599
  0.00203971  0.00418736  0.00169754  0.00446431  0.00448562  0.00610604
 -0.00320012 -0.00457704 -0.00042724  0.002535   -0.00326358  0.00605725
  0.00415338  0.00776569  0.00256863  0.00811646 -0.00138933  0.00807794
  0.00371757 -0.00804635 -0.00393261 -0.0024699   0.00489368 -0.00087089
 -0.00283206  0.00783478  0.00932223 -0.00161465 -0.00515883 -0.00470188
 -0.00484761 -0.00960413  0.00137302 -0.00422603  0.00252757  0.00561319
 -0.00406587 -0.00959716  0.00154623 -0.00670169  0.00249588 -0.00378196
  0.00707713  0.00064151  0.00356165 -0.0027375  -0

In [2]:
model.wv.most_similar("king")

[('the', 0.21617023646831512),
 ('a', 0.04469689726829529),
 ('queen', 0.001945548690855503),
 ('kingdom', -0.032872386276721954),
 ('man', -0.09325907379388809),
 ('is', -0.09575030952692032),
 ('woman', -0.10528077185153961),
 ('rules', -0.16934169828891754),
 ('strong', -0.17339195311069489)]

In [3]:
model.wv.most_similar(
    positive=["king", "woman"],
    negative=["man"]
)

[('a', 0.14207254350185394),
 ('the', 0.036040324717760086),
 ('kingdom', 0.01936185546219349),
 ('is', 0.016606055200099945),
 ('queen', 0.009253258816897869),
 ('strong', -0.003176614409312606),
 ('rules', -0.013764968141913414)]

# $ queen = king - man + woman $

In [4]:
king = model.wv["king"]
man = model.wv["man"]
woman = model.wv["woman"]

result = king - man + woman

print(result)

[-2.20548268e-02 -9.44156200e-05 -1.76382624e-02 -9.26324911e-03
 -6.97434042e-03 -8.56699049e-03 -3.99580412e-03  1.27755515e-02
 -3.80939827e-03 -4.10847133e-03 -2.70937430e-03 -4.76565585e-03
 -1.33007271e-02 -6.05467288e-03 -5.55056334e-03 -2.30728928e-03
 -6.99665351e-03 -1.55974720e-02  1.02565205e-02  7.97968707e-04
 -8.21433496e-03 -2.99578998e-03  1.58631755e-03  1.45499539e-02
  8.82649329e-03 -1.44623676e-02 -7.31493812e-04 -6.56631496e-03
  5.99920470e-03 -6.67412393e-03  2.08361223e-02 -7.76748406e-03
  8.64684116e-04  7.92833697e-03 -5.11659216e-03  1.56788938e-02
  1.34830810e-02  1.04681998e-02  1.43220322e-02 -1.42489281e-03
  4.96943807e-03  1.12284208e-03 -9.49942134e-03 -1.03518171e-02
 -9.86795500e-03  5.23708574e-03  3.03300447e-03  1.64798833e-02
 -5.68959536e-03  5.27086901e-03  1.39936460e-02  1.20563805e-03
 -3.00739706e-03  2.41621882e-02 -5.10018319e-04 -1.49199646e-02
  1.49423312e-02 -1.39667653e-02 -2.73984764e-03 -1.52720092e-02
 -2.74572335e-03  1.00011

In [5]:
model.wv.similar_by_vector(result, topn=5)

[('woman', 0.5870859026908875),
 ('king', 0.5351763963699341),
 ('a', 0.14069527387619019),
 ('the', 0.0310965683311224),
 ('kingdom', 0.022815953940153122)]

# From Scratch

In [6]:
text = "the king is a man the queen is a woman"

tokens = text.lower().split()

print(tokens)

['the', 'king', 'is', 'a', 'man', 'the', 'queen', 'is', 'a', 'woman']


In [7]:
import numpy as np

vocab = sorted(set(tokens))

word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for word, i in word_to_id.items()}

print(word_to_id)

{'a': 0, 'is': 1, 'king': 2, 'man': 3, 'queen': 4, 'the': 5, 'woman': 6}


In [8]:
window_size = 2

pairs = []

for i, target_word in enumerate(tokens):

    start = max(0, i - window_size)
    end = min(len(tokens), i + window_size + 1)

    for j in range(start, end):
        if i != j:
            pairs.append(
                (word_to_id[target_word],
                 word_to_id[tokens[j]])
            )
            
for pair in pairs:
    print(pair)

(5, 2)
(5, 1)
(2, 5)
(2, 1)
(2, 0)
(1, 5)
(1, 2)
(1, 0)
(1, 3)
(0, 2)
(0, 1)
(0, 3)
(0, 5)
(3, 1)
(3, 0)
(3, 5)
(3, 4)
(5, 0)
(5, 3)
(5, 4)
(5, 1)
(4, 3)
(4, 5)
(4, 1)
(4, 0)
(1, 5)
(1, 4)
(1, 0)
(1, 6)
(0, 4)
(0, 1)
(0, 6)
(6, 1)
(6, 0)


In [9]:
V = len(vocab)

In [10]:
def one_hot(word_id, vocab_size):
    vector = np.zeros(vocab_size)
    vector[word_id] = 1
    return vector

In [11]:
king_id = word_to_id["king"]

x = one_hot(king_id, V)

print(x)

[0. 0. 1. 0. 0. 0. 0.]


In [12]:
embedding_size = 10
learning_rate = 0.1
epochs = 1000

np.random.seed(42)

W1 = np.random.randn(V, embedding_size) * 0.01
W2 = np.random.randn(embedding_size, V) * 0.01

def softmax(x):
    x = x - np.max(x)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x)


for epoch in range(epochs):

    total_loss = 0

    for input_id, target_id in pairs:

        # Forward pass
        embedding = W1[input_id]

        score = embedding @ W2

        probabilities = softmax(score)

        loss = -np.log(probabilities[target_id] + 1e-9)

        total_loss += loss

        # Backward pass 
        target = np.zeros(V)
        target[target_id] = 1

        error = probabilities - target

        dW2 = np.outer(embedding, error)

        d_embedding = W2 @ error

        dW1 = np.zeros_like(W1)
        dW1[input_id] = d_embedding

        # Weights
        W1 -= learning_rate * dW1
        W2 -= learning_rate * dW2

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")


Epoch 0, Loss: 66.1605
Epoch 100, Loss: 59.1093
Epoch 200, Loss: 59.1411
Epoch 300, Loss: 59.1480
Epoch 400, Loss: 59.1506
Epoch 500, Loss: 59.1515
Epoch 600, Loss: 59.1517
Epoch 700, Loss: 59.1518
Epoch 800, Loss: 59.1518
Epoch 900, Loss: 59.1518


In [13]:
king_embedding = W1[word_to_id["king"]]

print(king_embedding)

[ 0.26796417 -0.9373978   0.29317405  0.73202197 -0.10111114  0.75691647
  0.24547925 -0.88541646  0.71003806  0.66410161]


In [14]:
def get_embedding(word):
    return W1[word_to_id[word]]

In [15]:
print(get_embedding("queen"))

[-0.73532009 -0.48142277  0.56073202  0.65324235  0.10433696  0.01869595
  0.6234721  -0.78553879  0.19602674  0.6179933 ]


In [16]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )

def most_similar(word, top_n=5):

    word_vector = get_embedding(word)
    similarities = []

    for other_word in vocab:

        if other_word == word:
            continue

        other_vector = get_embedding(other_word)

        similarity = cosine_similarity(
            word_vector,
            other_vector
        )

        similarities.append((other_word, similarity))

    similarities.sort(key=lambda x: x[1], reverse=True)

    return similarities[:top_n]


print(most_similar("king"))

[('woman', np.float64(0.7851409217122658)), ('queen', np.float64(0.6736960814947155)), ('man', np.float64(0.5672632367053972)), ('is', np.float64(-0.0875994488185116)), ('a', np.float64(-0.1172912043759922))]


In [17]:
king = get_embedding("king")
man = get_embedding("man")
woman = get_embedding("woman")

result = king - man + woman

In [18]:
result

array([ 0.16433593, -0.91516142,  1.57299402,  1.54128629, -0.97151952,
       -0.44557144,  0.25438308, -2.03230864,  1.03224321,  1.23861064])

In [19]:
woman

array([ 0.02452051, -0.63024259,  0.60441056,  0.6356431 , -0.53104435,
       -0.02308682, -0.40370517, -1.69160116,  0.62501158,  0.88275816])

In [20]:
def closest_words(vector, top_n=5):

    results = []

    for word in vocab:
        word_vector = get_embedding(word)

        similarity = cosine_similarity(vector, word_vector)

        results.append((word, similarity))

    results.sort(key=lambda x: x[1], reverse=True)

    return results[:top_n]


print(closest_words(result))

[('woman', np.float64(0.9240528348969864)), ('king', np.float64(0.7725947404585588)), ('queen', np.float64(0.7596851550738358)), ('the', np.float64(0.21551821043107114)), ('man', np.float64(0.019927218829187156))]
